In [5]:

from google.colab import files

uploaded = files.upload()


Saving Demographics_age_sex.xlsx to Demographics_age_sex (1).xlsx


In [6]:

import pandas as pd

file_path = "Demographics_age_sex.xlsx"

# Load every sheet from the original Excel workbook
sheets = pd.read_excel(file_path, sheet_name=None)

print("Number of sheets:", len(sheets))
print("\nSheet names:")
for name, df in sheets.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

# Inspect the main demographics/diagnosis sheet
display(sheets["Parselmouth"].head())


Number of sheets: 9

Sheet names:
Parselmouth: 81 rows × 4 columns
LPC_means: 81 rows × 12 columns
LAR_means: 81 rows × 12 columns
Cep_means: 81 rows × 12 columns
MFCC_means: 81 rows × 12 columns
LPC_vars: 81 rows × 12 columns
LAR_vars: 81 rows × 12 columns
Cep_vars: 81 rows × 12 columns
MFCC_vars: 81 rows × 12 columns


,Sample ID,Label,Age,Sex
0,AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5,HC,69.0,M
1,AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6,HC,43.0,M
2,AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2,HC,18.0,F
3,AH_123G_559F0706-2238-447C-BA39-DB5933BA619D,HC,28.0,M
4,AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22,HC,68.0,M


In [8]:

import pandas as pd

demo = sheets["Parselmouth"].copy()
demo.columns = demo.columns.astype(str).str.strip()

# Standardize the participant ID in the main sheet
demo = demo.rename(columns={"Sample": "Sample ID"})
df = demo.copy()

feature_sheets = [
    "LPC_means", "LAR_means", "Cep_means", "MFCC_means",
    "LPC_vars", "LAR_vars", "Cep_vars", "MFCC_vars"
]

for name in feature_sheets:
    feature_df = sheets[name].copy()
    feature_df.columns = feature_df.columns.astype(str).str.strip()

    # Feature sheets use 'Sample' as the participant ID
    if "Sample" in feature_df.columns:
        feature_df = feature_df.rename(columns={"Sample": "Sample ID"})
    elif "Sample ID" not in feature_df.columns:
        raise ValueError(f"ID column not found in {name}: {feature_df.columns.tolist()}")

    # Remove repeated label; the main sheet is the source of truth
    if "Label" in feature_df.columns:
        feature_df = feature_df.drop(columns=["Label"])

    # Clean IDs before merging
    feature_df["Sample ID"] = feature_df["Sample ID"].astype(str).str.strip()
    df["Sample ID"] = df["Sample ID"].astype(str).str.strip()

    df = df.merge(
        feature_df,
        on="Sample ID",
        how="inner",
        validate="one_to_one"
    )

# Validate the merged dataset
print("Merged dataset shape:", df.shape)
print("Unique participants:", df["Sample ID"].nunique())
print("Duplicate IDs:", df["Sample ID"].duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

print("\nDiagnosis distribution:")
print(df["Label"].value_counts())

display(df.head())


Merged dataset shape: (81, 84)
Unique participants: 81
Duplicate IDs: 0
Missing values: 0

Diagnosis distribution:
Label
HC      41
PwPD    40
Name: count, dtype: int64


,Sample ID,Label,Age,Sex,LPC1_x,LPC2_x,LPC3_x,LPC4_x,LPC5_x,LPC6_x,...,MFCC1_y,MFCC2_y,MFCC3_y,MFCC4_y,MFCC5_y,MFCC6_y,MFCC7_y,MFCC8_y,MFCC9_y,MFCC10_y
0,AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5,HC,69.0,M,1.569877,-0.983969,-0.091904,0.041568,0.191788,-0.288774,...,0.038758,0.022063,0.017446,0.040128,0.025198,0.075028,0.026978,0.022016,0.049959,0.016487
1,AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6,HC,43.0,M,1.941508,-1.604020,0.596933,-0.203145,0.049751,0.149339,...,0.376447,0.043026,0.113414,0.202135,0.076834,0.177133,0.106460,0.058875,0.070278,0.057396
2,AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2,HC,18.0,F,1.939210,-1.400189,0.267341,-0.117478,0.393466,-0.146857,...,0.181835,0.065868,0.081565,0.028965,0.032045,0.029428,0.045059,0.037183,0.020112,0.016252
3,AH_123G_559F0706-2238-447C-BA39-DB5933BA619D,HC,28.0,M,1.742062,-1.111498,0.209142,-0.208665,0.172042,0.230845,...,0.101589,0.012878,0.019212,0.015828,0.023448,0.048831,0.026585,0.024423,0.011855,0.016314
4,AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22,HC,68.0,M,1.370398,-0.725562,0.058470,-0.669424,0.762271,-0.077152,...,0.505912,0.031619,0.023236,0.051445,0.032242,0.031627,0.053048,0.027289,0.119403,0.046914


In [9]:

# Remove duplicate suffixes from feature names
# Keep the first occurrence if the same feature name appears more than once.
# We will verify duplicates before dropping anything.

print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())

# The expected 80 speech features should have unique names.
# Rebuild clean feature columns directly from the original sheets.
clean_df = sheets["Parselmouth"].copy()
clean_df.columns = clean_df.columns.astype(str).str.strip()

if "Sample" in clean_df.columns:
    clean_df = clean_df.rename(columns={"Sample": "Sample ID"})

for name in feature_sheets:
    feature_df = sheets[name].copy()
    feature_df.columns = feature_df.columns.astype(str).str.strip()

    if "Sample" in feature_df.columns:
        feature_df = feature_df.rename(columns={"Sample": "Sample ID"})

    # Keep participant ID and actual numeric feature columns only
    feature_cols = [
        col for col in feature_df.columns
        if col not in ["Sample ID", "Label"]
    ]
    feature_df = feature_df[["Sample ID"] + feature_cols]

    clean_df = clean_df.merge(
        feature_df,
        on="Sample ID",
        how="inner",
        validate="one_to_one"
    )

df = clean_df

# Encode labels: Parkinson's = 1, Healthy = 0
df["Target"] = df["Label"].map({"PwPD": 1, "HC": 0})

# Separate speech features from demographic/identifier columns
exclude_cols = ["Sample ID", "Label", "Age", "Sex", "Target"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].apply(pd.to_numeric, errors="raise")
y = df["Target"]

print("Clean dataset shape:", df.shape)
print("Number of speech features:", len(feature_cols))
print("Missing values in X:", X.isnull().sum().sum())
print("\nTarget distribution:")
print(y.value_counts().rename(index={0: "Healthy", 1: "Parkinson's"}))

display(X.head())


Duplicate column names: []
Clean dataset shape: (81, 85)
Number of speech features: 80
Missing values in X: 0

Target distribution:
Target
Healthy        41
Parkinson's    40
Name: count, dtype: int64


,LPC1_x,LPC2_x,LPC3_x,LPC4_x,LPC5_x,LPC6_x,LPC7_x,LPC8_x,LPC9_x,LPC10_x,...,MFCC1_y,MFCC2_y,MFCC3_y,MFCC4_y,MFCC5_y,MFCC6_y,MFCC7_y,MFCC8_y,MFCC9_y,MFCC10_y
0,1.569877,-0.983969,-0.091904,0.041568,0.191788,-0.288774,0.322170,-0.598547,0.407556,-0.107863,...,0.038758,0.022063,0.017446,0.040128,0.025198,0.075028,0.026978,0.022016,0.049959,0.016487
1,1.941508,-1.604020,0.596933,-0.203145,0.049751,0.149339,-0.181571,-0.076214,0.181712,-0.080028,...,0.376447,0.043026,0.113414,0.202135,0.076834,0.177133,0.106460,0.058875,0.070278,0.057396
2,1.939210,-1.400189,0.267341,-0.117478,0.393466,-0.146857,-0.267388,0.218443,0.052065,-0.110188,...,0.181835,0.065868,0.081565,0.028965,0.032045,0.029428,0.045059,0.037183,0.020112,0.016252
3,1.742062,-1.111498,0.209142,-0.208665,0.172042,0.230845,-0.125670,-0.521256,0.695029,-0.249518,...,0.101589,0.012878,0.019212,0.015828,0.023448,0.048831,0.026585,0.024423,0.011855,0.016314
4,1.370398,-0.725562,0.058470,-0.669424,0.762271,-0.077152,-0.350492,-0.027547,0.154888,-0.025133,...,0.505912,0.031619,0.023236,0.051445,0.032242,0.031627,0.053048,0.027289,0.119403,0.046914


In [10]:

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Reproducibility
SEED = 42

# Confirm dataset is ready
assert X.shape == (81, 80), f"Unexpected X shape: {X.shape}"
assert y.nunique() == 2, "Expected two diagnosis classes."
assert not X.isnull().any().any(), "X contains missing values."

# 5-fold stratified CV: preserves class proportions in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Feature selection happens inside each training fold to reduce leakage
k_features = 30

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000, class_weight="balanced", random_state=SEED
    ),
    "SVM (RBF)": SVC(
        kernel="rbf", C=1.0, class_weight="balanced",
        probability=True, random_state=SEED
    ),
    "SVM (Linear)": SVC(
        kernel="linear", C=1.0, class_weight="balanced",
        probability=True, random_state=SEED
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=SEED
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=SEED
    ),
    "LDA": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("selector", SelectKBest(score_func=f_classif, k=k_features)),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
        error_score="raise"
    )

    results.append({
        "Model": name,
        "Accuracy_mean": scores["test_accuracy"].mean(),
        "Accuracy_std": scores["test_accuracy"].std(),
        "Balanced_Accuracy": scores["test_balanced_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="Balanced_Accuracy", ascending=False
).reset_index(drop=True)

display(results_df.round(3))


,Model,Accuracy_mean,Accuracy_std,Balanced_Accuracy,Precision,Recall,F1,ROC_AUC
0,KNN,0.715,0.054,0.712,0.727,0.750,0.725,0.787
1,Extra Trees,0.665,0.097,0.664,0.663,0.675,0.667,0.719
2,Random Forest,0.651,0.119,0.650,0.675,0.625,0.646,0.695
3,LDA,0.651,0.125,0.650,0.700,0.525,0.600,0.666
4,SVM (RBF),0.640,0.096,0.638,0.667,0.650,0.647,0.724
5,SVM (Linear),0.639,0.130,0.638,0.673,0.575,0.617,0.644
6,Logistic Regression,0.590,0.099,0.589,0.606,0.575,0.586,0.653


In [11]:

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selector", SelectKBest(score_func=f_classif)),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid = [
    {
        "selector__k": [10, 20, 30, 40, 60, "all"],
        "model": [KNeighborsClassifier()],
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    {
        "selector__k": [10, 20, 30, 40, 60, "all"],
        "model": [SVC()],
        "model__kernel": ["rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__class_weight": [None, "balanced"]
    }
]

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

search.fit(X, y)

print("Best cross-validation balanced accuracy:",
      round(search.best_score_, 3))
print("\nBest parameters:")
print(search.best_params_)

best_model = search.best_estimator_


Best cross-validation balanced accuracy: 0.725

Best parameters:
{'model': KNeighborsClassifier(), 'model__n_neighbors': 3, 'model__p': 2, 'model__weights': 'uniform', 'selector__k': 30}
